In [74]:
import pandas as pd
from pathlib import Path
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns

import json
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize as sk_normalize

In [38]:
DATA_FOLDER = Path.cwd().parent / "artifacts" / "data_ingestion"

In [39]:
# Load each CSV, keep DataFrames in memory, and show .info()
csv_files = sorted(DATA_FOLDER.glob("*.csv"))

dataframes = {}
for csv_path in csv_files:
    name = csv_path.stem
    df = pd.read_csv(csv_path)
    dataframes[name] = df
    globals()[f"{name}_df"] = df
    print(f"\n{name}_df")
    df.info()



anime_df
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16206 entries, 0 to 16205
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         16206 non-null  int64 
 1   title      16206 non-null  object
 2   rating     16206 non-null  object
 3   synopsis   16206 non-null  object
 4   ep_bin     16206 non-null  object
 5   dur_bin    16206 non-null  object
 6   era        16206 non-null  object
 7   source_id  16206 non-null  int64 
 8   favorites  16206 non-null  int64 
 9   watching   16206 non-null  int64 
 10  completed  16206 non-null  int64 
dtypes: int64(5), object(6)
memory usage: 1.4+ MB

anime_genres_df
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48164 entries, 0 to 48163
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   anime_id  48164 non-null  int64
 1   genre_id  48164 non-null  int64
dtypes: int64(2)
memory usage: 752.7 KB

anime_prod

In [40]:
user_anime_ratings_df.isnull().sum()


user_id     0
anime_id    0
rating      0
dtype: int64

In [41]:
anime_df.head()

,id,title,rating,synopsis,ep_bin,dur_bin,era,source_id,favorites,watching,completed
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,10,22,1591,13652
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,11,0,12,0
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,10,0,4,336
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,10,0,7,267
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,10,0,16,255


Merging tables for content based recommender

In [42]:
df = anime_df.drop(columns=["favorites", "watching", "completed"])
df.head()

,id,title,rating,synopsis,ep_bin,dur_bin,era,source_id
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,10
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,11
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,10
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,10
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,10


In [43]:
genres_df.head()

,id,genre_name
0,0,Unknown
1,1,Action
2,2,Adventure
3,3,Cars
4,4,Comedy


In [44]:
producers_df.head()

,id,producer_name
0,0,Unknown
1,1,12 Diary Holders
2,2,1st PLACE
3,3,1theK
4,4,3xCube


In [45]:
studios_df.head()

,id,studio_name
0,0,Unknown
1,1,10Gauge
2,2,2:10 AM Animation
3,3,33 Collective
4,4,3xCube


In [46]:
sources_df.head()

,id,source_name
0,0,Unknown
1,1,4-koma manga
2,2,Book
3,3,Card game
4,4,Digital manga


In [47]:
anime_genres_df.head()

,anime_id,genre_id
0,7092,4
1,7092,27
2,7092,29
3,7092,36
4,33986,4


In [48]:
anime_producers_df.head()

,anime_id,producer_id
0,7092,0
1,33986,492
2,2920,0
3,1719,584
4,6293,514


In [49]:
anime_studios_df.head()

,anime_id,studio_id
0,7092,373
1,33986,0
2,2920,307
3,1719,320
4,6293,0


In [50]:
genres_agg = (
    anime_genres_df
    .merge(genres_df.rename(columns={"id": "genre_id"}), on="genre_id")
    .groupby("anime_id")["genre_name"]
    .apply(list)
    .reset_index(name="genres")
)

producers_agg = (
    anime_producers_df
    .merge(producers_df.rename(columns={"id": "producer_id"}), on="producer_id")
    .groupby("anime_id")["producer_name"]
    .apply(list)
    .reset_index(name="producers")
)

studios_agg = (
    anime_studios_df
    .merge(studios_df.rename(columns={"id": "studio_id"}), on="studio_id")
    .groupby("anime_id")["studio_name"]
    .apply(list)
    .reset_index(name="studios")
)

final_df = (
    df.rename(columns={"id": "anime_id"})
    .merge(
        sources_df.rename(columns={"id": "source_id", "source_name": "source"}),
        on="source_id",
        how="left",
    )
    .merge(genres_agg, on="anime_id", how="left")
    .merge(producers_agg, on="anime_id", how="left")
    .merge(studios_agg, on="anime_id", how="left")
    .drop(columns=["source_id"])
)

final_df.head()


,anime_id,title,rating,synopsis,ep_bin,dur_bin,era,source,genres,producers,studios
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,Original,"[Action, Drama, Mecha, Sci-Fi]","[Dentsu, AT-X, Ultra Super Pictures, Sony Musi...",[SANZIGEN]
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,Other,"[Comedy, Kids, Slice of Life]",[Unknown],[Unknown]
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,Original,[Dementia],[Unknown],[Unknown]
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,Original,[Dementia],[Studio Zero],[Unknown]
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,Original,"[Comedy, Magic]",[Unknown],[MooGoo]


In [52]:
len(final_df)

16206

In [53]:
for col in ["genres", "producers", "studios"]:
    final_df[col] = final_df[col].apply(
        lambda lst: [x for x in lst if x != "Unknown"] if isinstance(lst, list) else []
    )

final_df["source"] = final_df["source"].replace("Unknown", "")

final_df.head()


,anime_id,title,rating,synopsis,ep_bin,dur_bin,era,source,genres,producers,studios
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,2010s,Original,"[Action, Drama, Mecha, Sci-Fi]","[Dentsu, AT-X, Ultra Super Pictures, Sony Musi...",[SANZIGEN]
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,2020s,Other,"[Comedy, Kids, Slice of Life]",[],[]
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,2010s,Original,[Dementia],[],[]
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,2010s,Original,[Dementia],[Studio Zero],[]
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,2010s,Original,"[Comedy, Magic]",[],[MooGoo]


In [64]:
final_df['era'].value_counts()

era
2010s      8927
2000s      3112
90s        1704
classic    1695
2020s       768
Name: count, dtype: int64

In [65]:
mapping = {"2010s": "old", "2020s": "new", "2000s": "quite  old", "90s": "very old"}
final_df['era'] = final_df['era'].map(mapping)

User_based_df

In [55]:
users_df.head()

,user_id,email,username,password
0,2374,2374@gmail.com,test_2374,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
1,2375,2375@gmail.com,test_2375,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
2,2376,2376@gmail.com,test_2376,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
3,2377,2377@gmail.com,test_2377,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
4,2378,2378@gmail.com,test_2378,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...


In [56]:
user_anime_ratings_df.head()

,user_id,anime_id,rating
0,204,35756,5
1,204,8142,7
2,204,28833,4
3,204,28999,4
4,204,30485,4


In [60]:
u = set()

for user in user_anime_ratings_df["user_id"].unique():
    if user not in users_df["user_id"].values:
        u.add(user)
u

set()

Recommender: Content Based

In [66]:
final_df.head()

,anime_id,title,rating,synopsis,ep_bin,dur_bin,era,source,genres,producers,studios
0,33041,Bubuki Buranki: Hoshi no Kyojin,PG-13 - Teens 13 or older,Sequel of Bubuki Buranki .,short,standard,old,Original,"[Action, Drama, Mecha, Sci-Fi]","[Dentsu, AT-X, Ultra Super Pictures, Sony Musi...",[SANZIGEN]
1,42179,Lalalacoco II,G - All Ages,Second season of Lalalacoco.,movie_ova,standard,new,Other,"[Comedy, Kids, Slice of Life]",[],[]
2,31267,Akuma no Kairozu,G - All Ages,Film by Takashi Ito.,movie_ova,short_form,old,Original,[Dementia],[],[]
3,29681,Odoroki Ban,G - All Ages,Independent animation by Furukawa Taku.,movie_ova,short_form,old,Original,[Dementia],[Studio Zero],[]
4,18047,Seiyuu Deka,G - All Ages,girl who is protecting her city.,movie_ova,short_form,old,Original,"[Comedy, Magic]",[],[MooGoo]


In [69]:
FAISS_DIR = Path.cwd().parent / "faiss_artifacts"
FAISS_DIR.mkdir(parents=True, exist_ok=True)

def join_list(x):
    return " ".join(x) if isinstance(x, list) else ""

soup_df = final_df.copy()
soup_df["soup"] = (
    soup_df["title"].fillna("") + " " +
    soup_df["synopsis"].fillna("") + " " +
    soup_df["rating"].fillna("") + " " +
    soup_df["ep_bin"].fillna("") + " " +
    soup_df["dur_bin"].fillna("") + " " +
    soup_df["era"].fillna("") + " " +
    soup_df["source"].fillna("") + " " +
    soup_df["genres"].apply(join_list) + " " +
    soup_df["producers"].apply(join_list) + " " +
    soup_df["studios"].apply(join_list)
).str.lower()

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(soup_df["soup"])

vectors = normalize(tfidf_matrix, norm="l2", axis=1).astype(np.float32).toarray()

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

faiss.write_index(index, str(FAISS_DIR / "anime.index"))

anime_id_to_idx = {
    int(anime_id): int(i) for i, anime_id in enumerate(soup_df["anime_id"].tolist())
}
with open(FAISS_DIR / "anime_id_to_idx.json", "w") as f:
    json.dump(anime_id_to_idx, f)

import pickle
with open(FAISS_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print(f"vectors: {vectors.shape}, faiss ntotal: {index.ntotal}")


vectors: (16206, 20000), faiss ntotal: 16206


In [70]:
def recommend(anime_id: int, k: int = 10):
    idx = anime_id_to_idx.get(int(anime_id))
    if idx is None:
        raise ValueError(f"anime_id {anime_id} not found")

    query = vectors[idx:idx + 1]
    scores, neighbors = index.search(query, k + 1)

    idx_to_anime_id = {v: int(aid) for aid, v in anime_id_to_idx.items()}
    rows = []
    for score, nbr_idx in zip(scores[0], neighbors[0]):
        if nbr_idx == idx or nbr_idx == -1:
            continue
        rows.append({
            "anime_id": idx_to_anime_id[int(nbr_idx)],
            "similarity": float(score),
        })
        if len(rows) == k:
            break

    recs = pd.DataFrame(rows).merge(
        final_df[["anime_id", "title", "genres", "era"]],
        on="anime_id",
        how="left",
    )
    return recs


src = final_df.iloc[0]
print(f"Query: {src['title']} (id={src['anime_id']})")
recommend(src["anime_id"], k=10)


Query: Bubuki Buranki: Hoshi no Kyojin (id=33041)


,anime_id,similarity,title,genres,era
0,32023,0.485835,Bubuki Buranki,"[Action, Drama, Mecha, Sci-Fi]",old
1,28881,0.260825,New Initial D Movie: Legend 2 - Tousou,"[Cars, Seinen, Sports]",old
2,19613,0.252635,New Initial D Movie: Legend 1 - Kakusei,"[Action, Cars, Seinen, Sports]",old
3,39681,0.244113,D4DJ: First Mix,[Music],new
4,39619,0.243463,BanG Dream! Film Live,[Music],old
5,5089,0.228552,Noramimi 2,[Comedy],quite old
6,30952,0.224960,New Initial D Movie: Legend 3 - Mugen,"[Cars, Seinen, Sports]",old
7,39031,0.208178,B Rappers Street,"[Comedy, Music]",old
8,18247,0.197105,IS: Infinite Stratos 2,"[Action, Comedy, Ecchi, Harem, Mecha, Sci-Fi]",old
9,9488,0.195878,Cencoroll Connect,"[Action, Sci-Fi]",old


Recommmender: User Based

In [86]:
users_df.head()

,user_id,email,username,password
0,2374,2374@gmail.com,test_2374,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
1,2375,2375@gmail.com,test_2375,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
2,2376,2376@gmail.com,test_2376,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
3,2377,2377@gmail.com,test_2377,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...
4,2378,2378@gmail.com,test_2378,5e884898da28047151d0e56f8dc6292773603d0d6aabbd...


In [87]:
len(users_df)

310059

In [88]:
user_anime_ratings_df.head()

,user_id,anime_id,rating
0,204,35756,5
1,204,8142,7
2,204,28833,4
3,204,28999,4
4,204,30485,4


In [89]:
len(user_anime_ratings_df)

56726814

In [94]:
MIN_RATINGS = 0

anime_idx_map = {int(aid): i for i, aid in enumerate(soup_df["anime_id"].tolist())}

ratings = user_anime_ratings_df[
    user_anime_ratings_df["anime_id"].isin(anime_idx_map)
].copy()
ratings["anime_idx"] = ratings["anime_id"].map(anime_idx_map)

counts = ratings.groupby("user_id").size()
active_users = counts[counts >= MIN_RATINGS].index
ratings = ratings[ratings["user_id"].isin(active_users)]

user_means = ratings.groupby("user_id")["rating"].transform("mean")
ratings["centered"] = ratings["rating"] - user_means

user_ids = ratings["user_id"].unique()
user_idx_map = {int(uid): i for i, uid in enumerate(user_ids)}
ratings["user_idx"] = ratings["user_id"].map(user_idx_map)

n_users = len(user_ids)
n_anime = tfidf_matrix.shape[0]

R = csr_matrix(
    (ratings["centered"].astype(np.float32),
     (ratings["user_idx"].values, ratings["anime_idx"].values)),
    shape=(n_users, n_anime),
)

user_vectors_sparse = R @ tfidf_matrix
user_vectors_sparse = sk_normalize(user_vectors_sparse, norm="l2", axis=1)

dim = user_vectors_sparse.shape[1]
user_index = faiss.IndexFlatIP(dim)

CHUNK = 5000
for start in range(0, n_users, CHUNK):
    end = min(start + CHUNK, n_users)
    block = user_vectors_sparse[start:end].toarray().astype(np.float32)
    user_index.add(block)
    print(f"added {end}/{n_users}")

faiss.write_index(user_index, str(FAISS_DIR / "users.index"))

user_id_to_idx = {int(uid): int(i) for uid, i in user_idx_map.items()}
with open(FAISS_DIR / "user_id_to_idx.json", "w") as f:
    json.dump(user_id_to_idx, f)

print(f"user vectors: {n_users} × {dim}, faiss ntotal: {user_index.ntotal}")


added 5000/309484
added 10000/309484
added 15000/309484
added 20000/309484
added 25000/309484
added 30000/309484
added 35000/309484
added 40000/309484
added 45000/309484
added 50000/309484
added 55000/309484
added 60000/309484
added 65000/309484
added 70000/309484
added 75000/309484
added 80000/309484
added 85000/309484
added 90000/309484
added 95000/309484
added 100000/309484
added 105000/309484
added 110000/309484
added 115000/309484
added 120000/309484
added 125000/309484
added 130000/309484
added 135000/309484
added 140000/309484
added 145000/309484
added 150000/309484
added 155000/309484
added 160000/309484
added 165000/309484
added 170000/309484
added 175000/309484
added 180000/309484
added 185000/309484
added 190000/309484
added 195000/309484
added 200000/309484
added 205000/309484
added 210000/309484
added 215000/309484
added 220000/309484
added 225000/309484
added 230000/309484
added 235000/309484
added 240000/309484
added 245000/309484
added 250000/309484
added 255000/309484


In [95]:
avg_vec = np.asarray(user_vectors_sparse.mean(axis=0)).astype(np.float32)
avg_vec = avg_vec / (np.linalg.norm(avg_vec) + 1e-12)

avg_idx = user_index.ntotal
user_index.add(avg_vec.reshape(1, -1))

all_user_ids = set(users_df["user_id"].astype(int).tolist())
covered = set(user_id_to_idx.keys())
cold_users = all_user_ids - covered

for uid in cold_users:
    user_id_to_idx[int(uid)] = int(avg_idx)

faiss.write_index(user_index, str(FAISS_DIR / "users.index"))
with open(FAISS_DIR / "user_id_to_idx.json", "w") as f:
    json.dump(user_id_to_idx, f)

print(f"cold users mapped: {len(cold_users)} → faiss idx {avg_idx}")
print(f"faiss ntotal: {user_index.ntotal}, mapping size: {len(user_id_to_idx)}")


cold users mapped: 575 → faiss idx 309484
faiss ntotal: 309485, mapping size: 310059


In [96]:
309484 - 310059

-575

recommender: content flavoured

In [97]:
idx_to_user_id = {int(i): int(uid) for uid, i in user_idx_map.items()}

def recommend_cf(user_id: int,
                 k_neighbors: int = 50,
                 top_n: int = 10,
                 min_support: int = 3,
                 min_num_ratings: int = 50):
    u_idx = user_id_to_idx.get(int(user_id))
    if u_idx is None:
        raise ValueError(f"user_id {user_id} not found")

    # cold users all share avg_idx — neighbors would be meaningless
    if u_idx == avg_idx:
        raise ValueError(f"user_id {user_id} is a cold user; use a fallback recommender")

    query = user_vectors_sparse[u_idx].toarray().astype(np.float32)

    # +1 for self, +some slack in case neighbors include the avg-vector idx
    sims, nbr_idxs = user_index.search(query, k_neighbors + 5)
    sims, nbr_idxs = sims[0], nbr_idxs[0]

    neighbor_ids, neighbor_sims = [], []
    for s, ni in zip(sims, nbr_idxs):
        if ni == -1 or ni == u_idx or ni == avg_idx:
            continue
        nid = idx_to_user_id.get(int(ni))
        if nid is None:
            continue
        neighbor_ids.append(nid)
        neighbor_sims.append(float(s))
        if len(neighbor_ids) == k_neighbors:
            break

    sim_map = dict(zip(neighbor_ids, neighbor_sims))

    seen = set(
        user_anime_ratings_df.loc[
            user_anime_ratings_df["user_id"] == user_id, "anime_id"
        ].tolist()
    )

    nbr_ratings = user_anime_ratings_df[
        user_anime_ratings_df["user_id"].isin(sim_map)
        & ~user_anime_ratings_df["anime_id"].isin(seen)
    ].copy()
    nbr_ratings["sim"] = nbr_ratings["user_id"].map(sim_map)

    grouped = nbr_ratings.groupby("anime_id").agg(
        num_neighbors=("user_id", "size"),
        weighted_sum=("rating", lambda r: (r * nbr_ratings.loc[r.index, "sim"]).sum()),
        sim_sum=("sim", "sum"),
    )
    grouped = grouped[grouped["num_neighbors"] >= min_support]
    grouped["score"] = grouped["weighted_sum"] / grouped["sim_sum"]

    if min_num_ratings > 0 and "num_ratings" in ratings_df.columns:
        reliable = set(ratings_df.loc[ratings_df["num_ratings"] >= min_num_ratings, "anime_id"])
        grouped = grouped[grouped.index.isin(reliable)]

    recs = (
        grouped.sort_values("score", ascending=False)
        .head(top_n)
        .reset_index()
        .merge(
            final_df[["anime_id", "title", "genres", "era"]],
            on="anime_id",
            how="left",
        )
    )
    return recs[["anime_id", "title", "genres", "era", "score", "num_neighbors"]]


sample_uid = next(uid for uid, i in user_id_to_idx.items() if i != avg_idx)
print("sample user:", sample_uid)
recommend_cf(sample_uid, k_neighbors=50, top_n=10)


sample user: 204


,anime_id,title,genres,era,score,num_neighbors
0,820,Ginga Eiyuu Densetsu,"[Drama, Military, Sci-Fi, Space]",NaN,9.667296,3
1,28977,Gintama°,"[Action, Comedy, Historical, Parody, Samurai, ...",old,9.435036,7
2,37491,Gintama.: Shirogane no Tamashii-hen - Kouhan-sen,"[Action, Comedy, Historical, Parody, Samurai, ...",old,9.338958,3
3,12029,Uchuu Senkan Yamato 2199,"[Action, Drama, Military, Sci-Fi, Space]",old,9.188695,5
4,34096,Gintama.,"[Action, Comedy, Historical, Parody, Samurai, ...",old,9.176047,6
5,34537,Yoru wa Mijikashi Arukeyo Otome,"[Comedy, Romance]",old,9.002214,6
6,19,Monster,"[Drama, Horror, Mystery, Police, Psychological...",quite old,9.000017,19
7,3297,Aria the Origination,"[Fantasy, Sci-Fi, Shounen, Slice of Life]",quite old,9.000000,3
8,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Comedy, Drama, Fantasy, Ma...",quite old,8.978286,42
9,35180,3-gatsu no Lion 2nd Season,"[Drama, Game, Seinen, Slice of Life]",old,8.916066,23


recommender: hybrid

In [98]:
def recommend_hybrid(user_id: int,
                     k_neighbors: int = 50,
                     top_n: int = 10,
                     alpha: float = 0.6,
                     min_support: int = 3,
                     min_num_ratings: int = 50,
                     content_pool: int = 500):
    u_idx = user_id_to_idx.get(int(user_id))
    if u_idx is None:
        raise ValueError(f"user_id {user_id} not found")

    user_vec = user_vectors_sparse[u_idx].toarray().astype(np.float32)

    # ---------- CF side: neighbor-weighted ratings ----------
    is_cold = (u_idx == avg_idx)
    cf_scores = pd.Series(dtype="float32")

    if not is_cold:
        sims, nbr_idxs = user_index.search(user_vec, k_neighbors + 5)
        sims, nbr_idxs = sims[0], nbr_idxs[0]

        neighbor_ids, neighbor_sims = [], []
        for s, ni in zip(sims, nbr_idxs):
            if ni == -1 or ni == u_idx or ni == avg_idx:
                continue
            nid = idx_to_user_id.get(int(ni))
            if nid is None:
                continue
            neighbor_ids.append(nid)
            neighbor_sims.append(float(s))
            if len(neighbor_ids) == k_neighbors:
                break
        sim_map = dict(zip(neighbor_ids, neighbor_sims))

        seen = set(
            user_anime_ratings_df.loc[
                user_anime_ratings_df["user_id"] == user_id, "anime_id"
            ].tolist()
        )

        nbr = user_anime_ratings_df[
            user_anime_ratings_df["user_id"].isin(sim_map)
            & ~user_anime_ratings_df["anime_id"].isin(seen)
        ].copy()
        nbr["sim"] = nbr["user_id"].map(sim_map)
        nbr["w_rating"] = nbr["rating"] * nbr["sim"]

        agg = nbr.groupby("anime_id").agg(
            num_neighbors=("user_id", "size"),
            weighted_sum=("w_rating", "sum"),
            sim_sum=("sim", "sum"),
        )
        agg = agg[agg["num_neighbors"] >= min_support]
        cf_scores = (agg["weighted_sum"] / agg["sim_sum"]).rename("cf_score")
    else:
        seen = set()

    # ---------- Content side: cosine against anime index ----------
    c_scores_raw, c_idxs = index.search(user_vec, content_pool)
    c_scores_raw, c_idxs = c_scores_raw[0], c_idxs[0]

    idx_to_anime_id = {v: int(aid) for aid, v in anime_id_to_idx.items()}
    content_rows = []
    for s, ai in zip(c_scores_raw, c_idxs):
        if ai == -1:
            continue
        aid = idx_to_anime_id[int(ai)]
        if aid in seen:
            continue
        content_rows.append((aid, float(s)))
    content_scores = pd.Series(
        dict(content_rows), name="content_score", dtype="float32"
    )

    # ---------- Union, normalize, blend ----------
    candidates = pd.concat([cf_scores, content_scores], axis=1)

    if min_num_ratings > 0 and "num_ratings" in ratings_df.columns:
        reliable = set(ratings_df.loc[ratings_df["num_ratings"] >= min_num_ratings, "anime_id"])
        candidates = candidates[candidates.index.isin(reliable)]

    def minmax(s):
        s = s.astype("float32")
        lo, hi = s.min(skipna=True), s.max(skipna=True)
        if pd.isna(lo) or hi == lo:
            return s.fillna(0.0) * 0.0
        return ((s - lo) / (hi - lo)).fillna(0.0)

    candidates["cf_n"] = minmax(candidates["cf_score"])
    candidates["content_n"] = minmax(candidates["content_score"])

    effective_alpha = 0.0 if is_cold else alpha
    candidates["final_score"] = (
        effective_alpha * candidates["cf_n"]
        + (1 - effective_alpha) * candidates["content_n"]
    )

    out = (
        candidates.sort_values("final_score", ascending=False)
        .head(top_n)
        .reset_index()
        .rename(columns={"index": "anime_id"})
        .merge(final_df[["anime_id", "title", "genres", "era"]],
               on="anime_id", how="left")
    )
    return out[["anime_id", "title", "genres", "era",
                "final_score", "cf_score", "content_score"]]


sample_uid = next(uid for uid, i in user_id_to_idx.items() if i != avg_idx)
recommend_hybrid(sample_uid, alpha=0.6, top_n=10)


,anime_id,title,genres,era,final_score,cf_score,content_score
0,36999,Zoku Owarimonogatari,"[Comedy, Mystery, Supernatural, Vampire]",old,0.939474,8.792975,0.160279
1,32268,Koyomimonogatari,"[Comedy, Mystery, Supernatural]",old,0.833909,8.181863,0.140494
2,32915,Durarara!!x2 Ketsu: Dufufufu!!,"[Action, Mystery, Supernatural]",old,0.708629,7.313888,0.120105
3,5205,Kara no Kyoukai 7: Satsujin Kousatsu (Go),"[Action, Mystery, Romance, Supernatural, Thril...",quite old,0.707769,8.769116,0.088329
4,25537,Fate/stay night Movie: Heaven's Feel - I. Pres...,"[Action, Fantasy, Magic, Supernatural]",old,0.703057,8.207369,0.099018
5,33049,Fate/stay night Movie: Heaven's Feel - II. Los...,"[Action, Fantasy, Magic, Supernatural]",old,0.680712,8.697777,0.081412
6,11531,Un-Go: Inga-ron,"[Mystery, Supernatural]",old,0.664942,7.384393,0.104915
7,6624,Kara no Kyoukai Remix: Gate of Seventh Heaven,"[Action, Mystery, Romance, Super Power, Thriller]",quite old,0.643906,7.563033,0.094468
8,36862,Made in Abyss Movie 3: Fukaki Tamashii no Reimei,"[Adventure, Drama, Fantasy, Mystery, Sci-Fi]",new,0.627929,8.755595,0.063652
9,14807,Kara no Kyoukai: Mirai Fukuin,"[Drama, Mystery, Seinen, Supernatural]",old,0.620993,8.071742,0.076288
